# VLM-DENTAL — Chain-of-Thought Trace Generation & Verification

This notebook orchestrates autonomous CoT trace generation using **LangGraph** with local **vLLM** and a multi-provider **Verifier Pool** (NVIDIA NIM, Groq, OpenRouter, Gemini).

### Architecture Overview:
- **Cell 7a (Generate)**: Runs local `QuantTrio/Qwen3.5-9B-AWQ` via vLLM at maximum GPU speed, producing raw tool-augmented traces in `train_cot_traces_unverified.jsonl`.
- **Cell 7b (Verify)**: Verifies pending traces against ground truth using external LLM verifiers with round-robin fallback and 5-min cooldown rate limits, writing passing traces to `train_cot_traces.jsonl`.
- **Cell 8 (Auto-Push)**: Auto-pushes newly verified traces to your GitHub repository after each session.
- Both generate and verify support incremental resume.

## 1. Mount Google Drive & Setup Workspace
Mounts Drive to access persistent storage and clones/pulls the latest `VLM-DENTAL` repository.

In [ ]:
import os

# ============================================================
#  TOGGLE: Set IS_COLAB = True for Google Colab, False for PC
# ============================================================
IS_COLAB = True

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    work_dir = '/content/VLM-DENTAL'
    if not os.path.exists(work_dir):
        os.chdir('/content')
        os.system('git clone https://github.com/rezaxr14/VLM-DENTAL.git')
    os.chdir(work_dir)
    os.system('git pull')
else:
    # Local PC: assume you're already in the repo directory
    work_dir = os.getcwd()

print(f'Active working directory: {os.getcwd()}')
print(f'Mode: {"Google Colab" if IS_COLAB else "Local PC"}')

## 2. Install Dependencies
Installs `vLLM`, `langgraph`, `google-genai`, `ultralytics`, and repository dependencies.

> **Note on Runtime Restart:** If Colab prompts you with *"Restart runtime to use newly installed packages"*, click **Restart Session**, then skip this cell and continue directly from **Cell 3**.

In [ ]:
import sys
try:
    import vllm
    import langgraph
    import ultralytics
    print("✅ Dependencies already installed. Skipping pip install. (Fast boot!)")
except ImportError:
    print("⚠️ Missing dependencies detected. Running pip install...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"])
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[api]"])


## 3. Configure Credentials (.env & Colab Secrets)
Automatically loads credentials in two layers:
1. **`.env` file** — Auto-searches `/content/drive/MyDrive/VLM-DENTAL/.env`, the project root, or `/content/.env`.
2. **Colab Secrets Tab** — Any secret configured in the Colab Secrets tab (`google.colab.userdata`) automatically overrides `.env`.

In [ ]:
import os
import shutil
from pathlib import Path
from dotenv import load_dotenv

# Ensure correct working directory
candidate_dirs = [
    '/content/VLM-DENTAL',
    '/content/drive/MyDrive/VLM-DENTAL',
    '/content/drive/MyDrive/vlmdental',
    os.getcwd()
]
project_dir = next((d for d in candidate_dirs if d and os.path.exists(os.path.join(d, 'dental_agent'))), None)
if project_dir:
    os.chdir(project_dir)
print(f'Working directory: {os.getcwd()}')

# Auto-detect and load .env file
candidate_env_paths = [
    os.path.join(os.getcwd(), '.env'),
    '/content/drive/MyDrive/VLM-DENTAL/.env',
    '/content/drive/MyDrive/vlmdental/.env',
    '/content/drive/MyDrive/.env',
    '/content/VLM-DENTAL/.env',
    '/content/.env',
]

found_env = next((p for p in candidate_env_paths if os.path.exists(p) and os.path.getsize(p) > 0), None)
if found_env:
    local_env = os.path.join(os.getcwd(), '.env')
    if os.path.abspath(found_env) != os.path.abspath(local_env):
        shutil.copy(found_env, local_env)
        print(f'Copied .env from {found_env} to {local_env}')
    load_dotenv(local_env, override=True)
    print(f'Loaded credentials from: {found_env}')
else:
    if not os.path.exists('.env') and os.path.exists('.env.example'):
        shutil.copy('.env.example', '.env')
        print('Created .env from .env.example template')
    print('No populated .env found — checking Colab Secrets tab...')

# Layer 2: Overlay Colab Secrets (if configured)
try:
    from google.colab import userdata
    secret_keys = [
        'GEMINI_API_KEY', 'NVIDIA_API_KEY', 'GROQ_API_KEY', 'OPENROUTER_API_KEY',
        'NVIDIA_VERIFIER_MODEL', 'GROQ_VERIFIER_MODEL',
        'OPENROUTER_VERIFIER_MODEL', 'GEMINI_VERIFIER_MODEL',
        'GENERATOR_PROVIDER', 'GENERATOR_MODEL',
        'GENERATOR_COOLDOWN_SECONDS', 'GENERATOR_RPD_LIMIT',
        'NVIDIA_GENERATOR_MODEL', 'GROQ_GENERATOR_MODEL',
        'OPENROUTER_GENERATOR_MODEL', 'GEMINI_GENERATOR_MODEL',
        'API_COOLDOWN_SECONDS', 'API_RPD_LIMIT',
        'HF_TOKEN', 'GITHUB_TOKEN'
    ]
    overridden = []
    for key in secret_keys:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
                overridden.append(key)
        except Exception:
            pass
    if overridden:
        print(f'Colab Secrets injected {len(overridden)} variable(s): {overridden}')
except ImportError:
    pass

# Ensure sensible defaults for local trace generation
os.environ.setdefault('GENERATOR_PROVIDER', 'local')
os.environ.setdefault('GENERATOR_MODEL', 'Qwen/Qwen3.5-9B')
os.environ.setdefault('LOCAL_VLLM_BASE_URL', 'http://localhost:8000/v1')

# Print status overview
print('\n' + '=' * 50)
print('ACTIVE CONFIGURATION OVERVIEW')
print('=' * 50)
print(f"GENERATOR_PROVIDER : {os.environ.get('GENERATOR_PROVIDER')}")
print(f"GENERATOR_MODEL    : {os.environ.get('GENERATOR_MODEL')}")
has_github = bool(os.environ.get('GITHUB_TOKEN', '').strip() and not os.environ.get('GITHUB_TOKEN', '').startswith('your_'))
print(f"GITHUB_TOKEN       : {'SET' if has_github else 'NOT SET'}")
active_verifiers = [
    p for p in ['NVIDIA', 'GROQ', 'OPENROUTER', 'GEMINI']
    if os.environ.get(f'{p}_API_KEY', '').strip() and not os.environ.get(f'{p}_API_KEY', '').startswith('your_')
]
print(f"Active Verifiers   : {active_verifiers if active_verifiers else 'NONE (add API keys in .env or Secrets tab)'}")
print('=' * 50)

## 4. Model Cache Directory Setup
Configures `HF_HOME` for model weights. In Colab mode, uses the fast local SSD (`/content/`). On a local PC, uses `data/models/vllm_cache/`.

In [ ]:
import os

# In Colab: use fast local SSD. On PC: use persistent project path.
if IS_COLAB:
    vllm_cache_dir = '/content/local_vllm_cache'
else:
    vllm_cache_dir = os.path.join('data', 'models', 'vllm_cache')

os.makedirs(vllm_cache_dir, exist_ok=True)
os.environ['HF_HOME'] = vllm_cache_dir
os.environ['HF_HUB_CACHE'] = os.path.join(vllm_cache_dir, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(vllm_cache_dir, 'hub')

model_name = os.environ.get('GENERATOR_MODEL', 'QuantTrio/Qwen3.5-9B-AWQ')
hub_dir = os.path.join(vllm_cache_dir, 'hub')
model_slug = model_name.replace('/', '--')
cached_path = os.path.join(hub_dir, f'models--{model_slug}')

if os.path.exists(cached_path):
    cached_size_mb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, dn, fns in os.walk(cached_path) for f in fns
    ) / (1024 * 1024)
    print(f'Model cached at {cached_path} ({cached_size_mb:.0f} MB)')
else:
    print(f'Model not cached yet. vLLM will download {model_name} to {vllm_cache_dir} on startup.')
print(f'Cache location: {vllm_cache_dir}')

## 5. Download DENTEX Dataset (If Not Already Present)
Downloads and structures the DENTEX panoramic X-ray images needed for trace generation.

In [ ]:
# Download and structure DENTEX dataset
!python download_and_cleanup.py

In [ ]:
# Delete unused partial sub-datasets to save disk space
!rm -rf data/dentex/DENTEX/training_data/disease
!rm -rf data/dentex/DENTEX/training_data/quadrant
!rm -rf data/dentex/DENTEX/training_data/unlabelled/

!rm -rf data/dentex/DENTEX/testing_data/disease
!rm -rf data/dentex/DENTEX/testing_data/quadrant

# Remove any lingering temporary download zip/cache folders
!rm -rf hf_cache
print('Unused dataset subsets and cache cleaned up successfully!')

In [ ]:
import os

# Ensure data/traces directory exists
os.makedirs('data/traces', exist_ok=True)

# --- One-time: Rename legacy trace file to .old ---
legacy_path = 'data/traces/train_cot_traces.jsonl'
backup_path = 'data/traces/train_cot_traces.jsonl.old'
if os.path.exists(legacy_path) and not os.path.exists(backup_path):
    file_size = os.path.getsize(legacy_path)
    if file_size > 0:
        os.rename(legacy_path, backup_path)
        print(f'Renamed legacy traces ({file_size/1024:.0f}KB) to {backup_path}')
    else:
        print('Legacy trace file is empty, no rename needed.')
elif os.path.exists(backup_path):
    print(f'Legacy backup already exists at {backup_path}')

## 6. Launch Local vLLM Generator Server
Spawns the local OpenAI-compatible vLLM inference server for `Qwen/Qwen3.5-9B` and polls `/v1/models` until it is ready.

> **Note:** Skip this cell if `GENERATOR_PROVIDER` is set to an external API instead of `local`.

In [ ]:
import subprocess
import time
import os
import urllib.request

# 1. Kill any existing zombie vLLM processes to prevent OOM errors!
print('Cleaning up any orphaned vLLM processes from previous runs...')
os.system('pkill -f "vllm.entrypoints.openai.api_server"')
time.sleep(2)  # Give GPU time to release memory

os.system('pip uninstall -y torchaudio > /dev/null 2>&1')
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

model_name = os.environ.get('GENERATOR_MODEL', 'QuantTrio/Qwen3.5-9B-AWQ')
download_dir = os.path.join(vllm_cache_dir, 'hub')
port = '8000'
log_path = 'vllm_server.log'

print(f'Starting vLLM server for {model_name}')
print(f'Model download dir: {download_dir}')

log_file = open(log_path, 'w', encoding='utf-8')

vllm_cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', model_name,
    '--port', port,
    '--max-model-len', '32768',
    '--download-dir', download_dir,
    '--trust-remote-code',
    '--enforce-eager',
    '--dtype', 'auto',
]

if IS_COLAB:
    # T4: use pre-quantized AWQ checkpoint via Marlin kernels.
    # --max-num-seqs matters here: this AWQ build only quantizes the standard
    # linear layers (vision encoder + GDN attention blocks stay higher-precision),
    # so weights alone take ~11.2GB of the T4's 15GB, leaving very little headroom.
    # vLLM's default max-num-seqs (256) sizes the startup KV-cache/encoder-cache
    # profiling pass for way more concurrency than that headroom supports, which
    # is why startup silently hangs instead of erroring. We only process one
    # radiograph at a time anyway, so capping this costs nothing.
    vllm_cmd += ['--quantization', 'awq_marlin',
                 '--gpu-memory-utilization', '0.95',
                 '--max-num-seqs', '1']
else:
    # Local PC with larger GPU: full precision
    vllm_cmd += ['--gpu-memory-utilization', '0.95',
                 '--max-num-seqs', '1']

print(f'Quantization: {"AWQ (Marlin kernels)" if IS_COLAB else "none (fp16)"}')
vllm_process = subprocess.Popen(vllm_cmd, stdout=log_file, stderr=subprocess.STDOUT)

base_url = f'http://localhost:{port}/v1'
max_wait = 900
poll_interval = 5
elapsed = 0

print('Starting vLLM server... (Streaming logs live)')
with open(log_path, 'r', encoding='utf-8', errors='ignore') as log_reader:
    while elapsed < max_wait:
        ret_code = vllm_process.poll()
        if ret_code is not None:
            print(f'\n❌ vLLM process exited with code {ret_code}.')
            raise RuntimeError(f'vLLM server terminated with return code {ret_code}')

        # Stream any new log lines directly to Colab stdout
        new_logs = log_reader.read()
        if new_logs:
            print(new_logs, end='', flush=True)

        try:
            req = urllib.request.Request(f'{base_url}/models')
            with urllib.request.urlopen(req, timeout=3) as resp:
                if resp.getcode() == 200:
                    print(f'\n>> ✅ vLLM server is READY! (took {elapsed}s)')
                    break
        except Exception:
            pass
        time.sleep(poll_interval)
        elapsed += poll_interval
    else:
        print(f'\nERROR: vLLM server did not become ready within {max_wait}s.')
        raise RuntimeError('vLLM server startup timeout')


## 7a. Generate Raw Traces (GPU-Bound)
Runs the LangGraph agent reasoning loop on DENTEX images, saving raw diagnostic traces into `data/traces/train_cot_traces_unverified.jsonl`.

- Uses local `vLLM` with **zero API rate limits**.
- Fully resumable: automatically skips already processed images.

In [ ]:
!python scripts/run_trace_gen.py --mode generate --split train

## 7b. Verify Pending Traces (API-Bound)
Reads unverified traces from `train_cot_traces_unverified.jsonl` and validates diagnostic correctness against ground truth using external LLM verifiers.

- Passing traces are appended to `data/traces/train_cot_traces.jsonl`.
- Uses round-robin rotation across active verifier providers (NVIDIA NIM, Groq, OpenRouter, Gemini) with 5-minute cooldown.

In [ ]:
!python scripts/run_trace_gen.py --mode verify --split train

## 8. Auto-Push Verified Traces to GitHub
Commits and pushes newly verified trace datasets to GitHub using your configured `GITHUB_TOKEN`.

In [ ]:
import os
import subprocess

verified_path = 'data/traces/train_cot_traces.jsonl'
unverified_path = 'data/traces/train_cot_traces_unverified.jsonl'

has_verified = os.path.exists(verified_path) and os.path.getsize(verified_path) > 0
has_unverified = os.path.exists(unverified_path) and os.path.getsize(unverified_path) > 0
if not has_verified and not has_unverified:
    print('No traces to push yet. Run Trace Generation first.')
else:
    subprocess.run(['git', 'config', 'user.email', 'rezaxr14@gmail.com'])
    subprocess.run(['git', 'config', 'user.name', 'Reza Nadimi'])

    github_token = os.environ.get('GITHUB_TOKEN', '').strip()
    if not github_token or github_token.startswith('your_'):
        print('WARNING: GITHUB_TOKEN not configured. Please set it in .env or Colab Secrets tab.')
    else:
        result = subprocess.run(['git', 'remote', 'get-url', 'origin'], capture_output=True, text=True)
        remote_url = result.stdout.strip()
        if 'github.com' in remote_url and '@' not in remote_url:
            auth_url = remote_url.replace('https://github.com', f'https://{github_token}@github.com')
            subprocess.run(['git', 'remote', 'set-url', 'origin', auth_url])

        n_verified = 0
        if has_verified:
            with open(verified_path, 'r', encoding='utf-8') as f:
                n_verified = sum(1 for line in f if line.strip())
            subprocess.run(['git', 'add', verified_path])

        n_unverified = 0
        if has_unverified:
            with open(unverified_path, 'r', encoding='utf-8') as f:
                n_unverified = sum(1 for line in f if line.strip())
            subprocess.run(['git', 'add', unverified_path])

        status = subprocess.run(['git', 'diff', '--cached', '--name-only'], capture_output=True, text=True)
        if status.stdout.strip():
            msg_parts = []
            if n_verified > 0: msg_parts.append(f'{n_verified} verified')
            if n_unverified > 0: msg_parts.append(f'{n_unverified} unverified')
            commit_msg = f"data: update {' and '.join(msg_parts)} CoT traces (auto-push from Colab)"
            
            subprocess.run(['git', 'commit', '-m', commit_msg])
            push_result = subprocess.run(['git', 'push'], capture_output=True, text=True)
            if push_result.returncode == 0:
                print(f'Successfully pushed to GitHub: {commit_msg}')
            else:
                print(f'Push failed: {push_result.stderr}')
        else:
            print('Traces are already up-to-date on GitHub.')


## 9. Status Dashboard & Pool Capacity
Inspects total generated vs verified traces and current API rate limit capacity.

In [ ]:
import json
import os

unverified_path = 'data/traces/train_cot_traces_unverified.jsonl'
verified_path = 'data/traces/train_cot_traces.jsonl'

def count_ids(path):
    ids = set()
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    r = json.loads(line.strip())
                    if 'image_id' in r:
                        ids.add(int(r['image_id']))
                except Exception:
                    pass
    return ids

unverified_ids = count_ids(unverified_path)
verified_ids = count_ids(verified_path)
pending = unverified_ids - verified_ids

print('=' * 50)
print('TRACE GENERATION STATUS DASHBOARD')
print('=' * 50)
print(f'Generated (unverified) : {len(unverified_ids)}')
print(f'Verified (SFT-ready)   : {len(verified_ids)}')
print(f'Pending verification   : {len(pending)}')
print('=' * 50)
print()
!python scripts/run_trace_gen.py --mode verify --status-only

## 10. Manual Download Helper
Downloads verified trace files directly to your local computer via browser.

In [ ]:
from google.colab import files
import os

verified_path = 'data/traces/train_cot_traces.jsonl'
unverified_path = 'data/traces/train_cot_traces_unverified.jsonl'

for path in [verified_path, unverified_path]:
    if os.path.exists(path) and os.path.getsize(path) > 0:
        files.download(path)
        print(f'Downloading {path}...')
    else:
        print(f'No traces found at {path}')
